In [22]:
import os
import weaviate
from weaviate.classes.init import Auth
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_weaviate import WeaviateVectorStore
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_cohere import CohereRerank
from langchain_core.runnables import RunnableSequence, RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

In [2]:
# Helper function for printing docs

def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            [f"Document {i + 1}:\n\n" + d.page_content for i, d in enumerate(docs)]
        )
    )

In [ ]:
loader = PyPDFLoader(r'papers/RAG-for-NLP_paper.pdf')
docs = loader.load()

In [12]:
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
chunks = splitter.split_documents(docs)

In [18]:
# Clean invalid metadata keys for Weaviate
# Weaviate does not allow "." in property names

for doc in chunks:
    cleaned_metadata = {}

    for key, value in doc.metadata.items():
        # Replace dots with underscores
        clean_key = key.replace(".", "_")

        cleaned_metadata[clean_key] = value

    # Update document metadata
    doc.metadata = cleaned_metadata

In [3]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.getenv("WEAVIATE_URL"),
    auth_credentials=Auth.api_key(os.getenv("WEAVIATE_API_KEY"))
)

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="thenlper/gte-small"
)

vector_store = WeaviateVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    client=client,
    index_name="RAGDocs"
)

collection = client.collections.get("RAGDocs")
print(collection.config.get())

c:\Users\RadhaKrishna\Documents\Lib\site-packages\weaviate\warnings.py:302: ResourceWarning: Con004: The connection to Weaviate was not closed properly. This can lead to memory leaks.
            Please make sure to close the connection using `client.close()`.
  warnings.warn(
C:\Users\RadhaKrishna\AppData\Local\Temp\ipykernel_1084\2176823988.py:10: ResourceWarning: unclosed <ssl.SSLSocket fd=8880, family=2, type=1, proto=0, laddr=('10.143.215.173', 65297), raddr=('34.160.174.119', 443)>
  vector_store = WeaviateVectorStore.from_documents(


In [ ]:
# Reconnect to the existing Weaviate vector store without reinserting documents
embeddings = HuggingFaceEmbeddings(
    model_name="thenlper/gte-small"
)

vector_store = WeaviateVectorStore(
    client=client,
    index_name="RAGDocs",
    embedding=embeddings,
    text_key="text"
)

In [4]:
# As chunks are already ingested in the Vector Database, just connect to it
client.connect()

In [10]:
print(client.is_ready())

True


In [27]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)

In [12]:
retriever = vector_store.as_retriever(
    search_kwargs={
        "k": 5,
        "alpha": 0.5       # Hybrid Search
    }
)
retriever

VectorStoreRetriever(tags=['WeaviateVectorStore', 'HuggingFaceEmbeddings'], vectorstore=<langchain_weaviate.vectorstores.WeaviateVectorStore object at 0x0000019AB37C8380>, search_kwargs={'k': 5, 'alpha': 0.5})

In [18]:
compressor = CohereRerank(model="rerank-v4.0-pro", top_n=2)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever
)

print(pretty_print_docs(compression_retriever.invoke("What are Aliens?")))

Document 1:

beyond. Found. Trends Inf. Retr., 3(4):333–389, April 2009. ISSN 1554-0669. doi: 10.1561/
1500000019. URL https://doi.org/10.1561/1500000019.
----------------------------------------------------------------------------------------------------
Document 2:

ard of wikipedia: Knowledge-powered conversational agents. In International Conference on
Learning Representations, 2019. URL https://openreview.net/forum?id=r1l73iRqKm.
None


In [19]:
prompt = PromptTemplate.from_template(
    """
Answer ONLY from the provided context.

If the context does not contain the answer,
say exactly:
"I do not have relevant information in the context."

Do not use outside knowledge.
Do not speculate.

Context:
{context}

Question:
{question}

Answer:
"""
)

In [15]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [28]:
parallel_chain = RunnableParallel({
    'context': RunnableSequence(compression_retriever, RunnableLambda(format_docs)),
    'question': RunnablePassthrough()
})

main_chain = parallel_chain | prompt | llm | StrOutputParser()

In [29]:
print(main_chain.invoke("What are Aliens?"))

I do not have relevant information in the context.


In [30]:
query = """
Explain Jeopardy-style Question Answering in Natural Language Processing (NLP).

Also compare it with Abstractive Question Answering by discussing:

1. How each approach works
2. Whether answers are retrieved or generated
3. Their dependence on external knowledge retrieval
4. Their factual reliability and hallucination risk
5. Their strengths and weaknesses in real-world NLP applications

Finally, based strictly on the provided context, explain which approach is considered more effective and why.
"""

print(main_chain.invoke(query))

Based on the provided context:

**Jeopardy-style Question Answering in Natural Language Processing (NLP):**
Jeopardy-style Question Answering, as described in the context, involves "Jeopardy Question Generation: Answer Query". This means the system is given an "Answer Query" (an answer) and its task is to generate the corresponding question.

**Comparison with Abstractive Question Answering:**

**1. How each approach works**
*   **Jeopardy-style QA:** This approach works by taking an "Answer Query" as input and generating a question as the output.
*   **Abstractive QA:** This approach uses RAG models to "go beyond simple extractive QA and answer questions with free-form, abstractive" responses.

**2. Whether answers are retrieved or generated**
*   **Jeopardy-style QA:** The output, which is a question, is "generated" as indicated by "Question Generation".
*   **Abstractive QA:** Answers are "generated" in a "free-form, abstractive" manner. The mention of "RAG models" (Retrieval-Augmen

In [31]:
# To avoid data leakage
client.close()